<a href="https://colab.research.google.com/github/nat5natalia/butterfly_classifier/blob/master/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Butterfly Classifier

This notebook trains the butterfly classifier using data prepared in the project.
It assumes:
- The project folder is uploaded to Google Drive (e.g., in `MyDrive/butterfly_classifier/`)
- The data (images) are in `data/raw/` and splits CSV files are present.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# Путь к проекту в Drive (измените, если нужно)
PROJECT_PATH = '/content/drive/MyDrive/butterfly_classifier'
os.chdir(PROJECT_PATH)

# Устанавливаем зависимости, если их нет
!pip install -r requirements.txt

In [ ]:
import sys
sys.path.append(PROJECT_PATH)

import tensorflow as tf
from src.config import CFG
from src.preprocessing import create_dataset
from src.train import create_model, compile_model, train_model, save_model
from src.utils import load_json

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

In [ ]:
!ls data/splits/
!ls data/raw/

In [ ]:
class_map = load_json(CFG.class_map_json)
num_classes = len(class_map)
print("Classes:", class_map)
print("Number of classes:", num_classes)

In [ ]:
train_ds = create_dataset(CFG.train_csv, class_map, shuffle=True, augment=True)
val_ds = create_dataset(CFG.val_csv, class_map, shuffle=False, augment=False)

# Проверка
for images, labels in train_ds.take(1):
    print(images.shape, labels.shape)

In [ ]:
model = create_model(num_classes)
compile_model(model)
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        'models/best_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max'
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2,
        min_lr=1e-6
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=CFG.epochs,
    callbacks=callbacks
)

In [ ]:
save_model(model, 'models/butterfly_model.keras')
print("Model saved to", CFG.models_dir / "butterfly_model.keras")

In [ ]:
import matplotlib.pyplot as plt

def plot_history(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history.history['accuracy'], label='train')
    ax1.plot(history.history['val_accuracy'], label='val')
    ax1.set_title('Accuracy')
    ax1.legend()

    ax2.plot(history.history['loss'], label='train')
    ax2.plot(history.history['val_loss'], label='val')
    ax2.set_title('Loss')
    ax2.legend()
    plt.show()

plot_history(history)

In [ ]:
test_ds = create_dataset(CFG.test_csv, class_map, shuffle=False, augment=False)
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.4f}")